Vamos a desarrollar un caso de uso para el método de selección de variables Backward Elimination, aplicado a la regresión lineal.

Este método comienza con un modelo que incluye todas las variables disponibles y luego elimina sucesivamente la variable menos significativa (aquella cuya ausencia mejora más el rendimiento del modelo o su eliminación afecta menos el rendimiento) hasta que todas las variables restantes son significativas según un criterio predefinido.

Para este ejemplo, utilizaremos el conjunto de datos de California Housing disponible en Scikit-learn, que contiene información sobre diferentes aspectos de las viviendas en el área de California.

Nuestro objetivo será construir un modelo de regresión lineal para predecir el precio medio de las viviendas a partir de estas características, utilizando Backward Elimination para seleccionar las variables más relevantes.

In [8]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

In [12]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)

###Implementar Backward Elimination
La idea es comenzar con todas las variables y eliminar una a una las menos significativas basándonos en el p-valor de los coeficientes.

Utilizaremos un umbral de p-valor de 0.05 para decidir si una variable debe ser eliminada.

In [14]:
X_train_const = sm.add_constant(X_train)  # Añadir una constante al modelo
model = sm.OLS(y_train, X_train_const).fit()  # Ajustar el modelo con todas las variables
while True:
    # Buscar la variable con el mayor p-valor
    max_p_value = max(model.pvalues)
    feature_with_max_p = model.pvalues.idxmax()
    if max_p_value > 0.05:
        # Si el p-valor es mayor que el umbral, eliminar la variable y ajustar de nuevo
        X_train_const = X_train_const.drop(columns=[feature_with_max_p])
        model = sm.OLS(y_train, X_train_const).fit()
    else:
        # Si todos los p-valores son menores que el umbral, terminar el loop
        break

In [15]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.608
Model:                            OLS   Adj. R-squared:                  0.608
Method:                 Least Squares   F-statistic:                     3656.
Date:                Tue, 19 Mar 2024   Prob (F-statistic):               0.00
Time:                        20:03:02   Log-Likelihood:                -18077.
No. Observations:               16512   AIC:                         3.617e+04
Df Residuals:                   16504   BIC:                         3.623e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -37.1880      0.733    -50.710      0.0

Este código realiza la selección de variables mediante Backward Elimination, comenzando con un modelo que incluye todas las posibles variables explicativas y eliminando sucesivamente la variable menos significativa en cada paso. E

El loop termina cuando todas las variables restantes en el modelo tienen un p-valor menor que el umbral establecido (en este caso, 0.05), indicando que todas son estadísticamente significativas.

La salida del modelo (model.summary()) proporciona un resumen completo del modelo final, incluyendo las variables que quedaron, sus coeficientes, p-valores, y otras estadísticas importantes del modelo, como el R-cuadrado ajustado, que indica qué tan bien el modelo explica la variabilidad de los datos observados.

### Para entender el Model Summary
Un summary típico de statsmodels para un modelo de regresión lineal incluye varias secciones importantes:

1. **Model Summary**

Dep. Variable: La variable dependiente o el objetivo que estás intentando predecir.

Model: El tipo de modelo utilizado; por ejemplo, OLS indica una regresión lineal ordinaria.

Method: El método utilizado para ajustar el modelo, como los mínimos cuadrados.
No. Observations: El número de observaciones utilizadas en el modelo.

DF Residuals: Grados de libertad de los residuos; básicamente, el número de observaciones menos el número de parámetros estimados (incluyendo el intercepto).

DF Model: Número de parámetros en el modelo (sin contar el término constante).

2. **Coefficients Table**

coef: El valor estimado del coeficiente para cada variable explicativa y el término constante.

std err: El error estándar de la estimación del coeficiente.

t: El valor de t-statistic para la prueba de hipótesis de que el coeficiente es diferente de 0.

P > |t|: El p-valor asociado con la prueba de hipótesis. Un valor bajo indica que hay una fuerte evidencia contra la hipótesis nula, sugiriendo que la variable es significativa para el modelo.

[0.025 0.975]: El intervalo de confianza al 95% para el coeficiente, indicando el rango dentro del cual estamos confiados que el verdadero valor del coeficiente cae.

3. **Model Fit Statistics**

R-squared: Una medida de cuánta varianza en la variable dependiente es explicada por el modelo. Varía de 0 a 1, con valores más altos indicando un mejor ajuste.

Adjusted R-squared: Una versión ajustada del R-cuadrado que toma en cuenta el número de predictores en el modelo. Útil para comparar modelos con diferentes números de variables independientes.

F-statistic: Una prueba que evalúa la significancia global del modelo. Un valor alto sugiere que al menos una de las variables explicativas es significativa.
Prob (F-statistic): El p-valor asociado con la F-statística.

Log-Likelihood: El logaritmo de la verosimilitud del modelo; útil para comparar modelos con diferentes números de parámetros.

AIC: El Criterio de Información de Akaike, una medida de la calidad del modelo que penaliza la complejidad para evitar el sobreajuste.

BIC: El Criterio de Información Bayesiano, otra medida que penaliza la complejidad del modelo pero de manera más estricta que el AIC.

Cuando interpretes el summary, presta especial atención a los p-valores de las variables para determinar su significancia, el R-cuadrado ajustado para evaluar el ajuste del modelo y el AIC/BIC para comparar modelos. La significancia de las variables te ayudará a entender cuáles son los predictores más importantes de tu variable objetivo.